[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/colab_homography_probe.ipynb)

# Feasibility probe — temporal homography propagation on Trace

**Question:** Trace is a virtual-PTZ crop of a *fixed* camera (no parallax), so
consecutive frames are related by a global homography. Can we therefore bridge
the ~84% of frames that lack pitch keypoints by registering them to the ~16%
that *do* (anchors), instead of needing landmarks every frame?

**Ground-truthed test:** for each pair of consecutive landmark anchors A → B
with a no-landmark gap between them, chain frame-to-frame homographies across the
gap to propagate A's pitch homography to B, then check the propagated map against
B's *own* detected landmarks. The error (in pitch-units, 0–1) says whether
propagation reproduces a real landmark fit.

- Players are masked out when estimating camera motion (they aren't static).
- Re-anchoring on B is the "loop closure" that would bound drift in a full build.

**Read:** pitch-error < ~0.03 ≈ excellent (~2 m), < ~0.05 good, > ~0.10 poor.
If most gaps come back small, propagation is the fix and we design it properly.
If registration is fragile on fast youth pans, we fall back to the detector
fine-tune. Needs the Stage-1 checkpoint at `/content/out/` (from the pipeline
demo) and the clip on Drive. No GPU, no labeling.

In [ ]:
# Install (skip if running in the same session as the pipeline demo).
!rm -rf /content/soccer-vision
!git clone -q https://github.com/PatrickJReed/soccer-vision.git /content/soccer-vision
!pip install -q "/content/soccer-vision/packages/soccer-vision[roboflow]"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from soccer_vision.pitch.landmarks import PITCH_LANDMARKS, build_frame_homographies

OUT = Path("/content/out")
CLIP = "/content/drive/MyDrive/soccer-vision/bakeoff_clip.mp4"

kp = pd.read_parquet(OUT / "keypoints.parquet")
traj = pd.read_parquet(OUT / "trajectories_px.parquet")

# Anchor frames: pixel->pitch homography from detected landmarks.
H_pitch = build_frame_homographies(kp, conf_threshold=0.5)
anchors = sorted(H_pitch)
print(f"{len(anchors)} anchor frames ({len(anchors) / 3733:.1%} of clip)")
gaps = [(anchors[i], anchors[i + 1], anchors[i + 1] - anchors[i]) for i in range(len(anchors) - 1)]
testable = [g for g in gaps if 2 <= g[2] <= 90]
print(f"{len(testable)} testable gaps (2..90 frames). Sample:", testable[:6])

In [ ]:
orb = cv2.ORB_create(3000)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)


def _player_mask(frame_idx, shape):
    """255 on static background, 0 over player/ref boxes (dilated) — so registration
    keys on the field/goal/background, not moving people."""
    m = np.full(shape[:2], 255, np.uint8)
    sel = traj[(traj.frame == frame_idx) & traj["class"].isin(["player", "goalkeeper", "referee"])]
    for _, r in sel.iterrows():
        cv2.rectangle(m, (int(r.bbox_x1) - 12, int(r.bbox_y1) - 12),
                      (int(r.bbox_x2) + 12, int(r.bbox_y2) + 12), 0, -1)
    return m


def interframe_homography(fa, frame_a, fb, frame_b):
    """Homography mapping frame_a pixels -> frame_b pixels (static background only)."""
    ga = cv2.cvtColor(frame_a, cv2.COLOR_BGR2GRAY)
    gb = cv2.cvtColor(frame_b, cv2.COLOR_BGR2GRAY)
    ka, da = orb.detectAndCompute(ga, _player_mask(fa, frame_a.shape))
    kb, db = orb.detectAndCompute(gb, _player_mask(fb, frame_b.shape))
    if da is None or db is None or len(da) < 12 or len(db) < 12:
        return None, 0
    matches = bf.match(da, db)
    if len(matches) < 12:
        return None, len(matches)
    src = np.float32([ka[m.queryIdx].pt for m in matches])
    dst = np.float32([kb[m.trainIdx].pt for m in matches])
    G, inliers = cv2.findHomography(src, dst, cv2.RANSAC, 3.0)
    return G, (int(inliers.sum()) if inliers is not None else 0)

In [ ]:
cap = cv2.VideoCapture(CLIP)
results = []  # (A, B, gap, error, min_inliers)

for A, B, gap in testable:
    cap.set(cv2.CAP_PROP_POS_FRAMES, A)
    ok, prev = cap.read()
    if not ok:
        continue
    W = np.eye(3)          # accumulates: maps frame-A pixels -> frame-f pixels
    min_inl = 1_000_000
    good = True
    for f in range(A + 1, B + 1):
        ok, cur = cap.read()
        if not ok:
            good = False
            break
        G, n_inl = interframe_homography(f - 1, prev, f, cur)
        if G is None:
            good = False
            break
        W = G @ W
        min_inl = min(min_inl, n_inl)
        prev = cur
    if not good:
        results.append((A, B, gap, None, 0))
        continue
    # Propagated pitch homography at B: pitch = H_A @ inv(W) @ pixel_B
    H_B_prop = H_pitch[A] @ np.linalg.inv(W)
    kb = kp[(kp.frame == B) & (kp.conf >= 0.5) & (kp.kp_idx < len(PITCH_LANDMARKS))]
    pts = np.column_stack([kb.x_px.to_numpy(), kb.y_px.to_numpy(), np.ones(len(kb))])
    mapped = (H_B_prop @ pts.T).T
    mapped /= mapped[:, 2:3]
    target = PITCH_LANDMARKS[kb.kp_idx.to_numpy()]
    err = float(np.linalg.norm(mapped[:, :2] - target, axis=1).mean())
    results.append((A, B, gap, err, min_inl))

cap.release()

print(f"{'A':>5} {'B':>5} {'gap':>4} {'min_inl':>8}  pitch-error")
for A, B, gap, err, inl in results:
    e = "FAIL(registration)" if err is None else f"{err:.3f}"
    print(f"{A:5d} {B:5d} {gap:4d} {inl:8d}  {e}")

errs = [e for *_, e, _ in results if e is not None]
n_fail = sum(1 for *_, e, _ in results if e is None)
print("\n--- verdict ---")
print(f"gaps tested: {len(results)}   registration failures: {n_fail}")
if errs:
    errs_sorted = sorted(errs)
    med = errs_sorted[len(errs_sorted) // 2]
    print(f"pitch-error  median {med:.3f}   "
          f"<=0.05: {sum(e <= 0.05 for e in errs) / len(errs):.0%}   "
          f"<=0.10: {sum(e <= 0.10 for e in errs) / len(errs):.0%}")

## Visual sanity — one propagated gap frame

Take a mid-gap frame (no landmarks of its own), propagate the homography to it,
and scatter its players on the canonical pitch. If propagation works, players
land in `[0,1]²` and look like a plausible formation — coverage we currently
throw away.

In [ ]:
good = [r for r in results if r[3] is not None and r[3] <= 0.10]
if not good:
    print("No gap registered well enough to visualize; see the table above.")
else:
    import matplotlib.pyplot as plt
    A, B, gap, err, _ = good[len(good) // 2]
    mid = (A + B) // 2

    cap = cv2.VideoCapture(CLIP)
    cap.set(cv2.CAP_PROP_POS_FRAMES, A)
    ok, prev = cap.read()
    W = np.eye(3)
    for f in range(A + 1, mid + 1):
        ok, cur = cap.read()
        G, _ = interframe_homography(f - 1, prev, f, cur)
        W = G @ W
        prev = cur
    cap.release()
    H_mid = H_pitch[A] @ np.linalg.inv(W)

    players = traj[(traj.frame == mid) & traj["class"].isin(["player", "goalkeeper"])]
    pts = np.column_stack([players.x_px.to_numpy(), players.y_px.to_numpy(), np.ones(len(players))])
    mapped = (H_mid @ pts.T).T
    mapped /= mapped[:, 2:3]

    plt.figure(figsize=(6, 4))
    colors = {"own": "tab:blue", "opp": "tab:red", "unknown": "gray"}
    for team in players["team"].unique():
        m = players["team"].to_numpy() == team
        plt.scatter(mapped[m, 0], mapped[m, 1], c=colors.get(team, "k"), label=team, s=60)
    plt.axhline(0.333, ls=":", c="gray"); plt.axhline(0.667, ls=":", c="gray")
    plt.xlim(-0.1, 1.1); plt.ylim(-0.1, 1.1); plt.gca().invert_yaxis()
    plt.xlabel("x_pitch"); plt.ylabel("y_pitch (goal-to-goal)")
    plt.title(f"frame {mid} (no own landmarks) — propagated from anchor {A}, gap {gap}, err {err:.3f}")
    plt.legend(); plt.tight_layout(); plt.show()